<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/nlp_sentiment/nlp_step03_IMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# 1. 데이터 불러오기
# Hugging Face datasets 라이브러리 사용
# !pip install datasets

import os

if not os.path.exists('aclImdb'):
  !wget -q http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
  !tar -xzf aclImdb_v1.tar.gz
  print("다운로드 완료")

# 파일에서 리뷰텍스트 읽어오기
def load_imdb(path, split='train'):
  texts, labels = [], []
  for label, sentiment in enumerate(['neg', 'pos']):
    folder = os.path.join(path, split, sentiment)
    for fname in os.listdir(folder):
      with open(os.path.join(folder, fname), encoding='utf-8') as f:
        texts.append(f.read())
      labels.append(label)
  return texts, labels

train_texts, train_labels = load_imdb('aclImdb', 'train')
test_texts, test_labels = load_imdb('aclImdb', 'test')

df_train = pd.DataFrame({'text': train_texts, 'label': train_labels})
df_test = pd.DataFrame({'text': test_texts, 'label': test_labels})

print(df_train.shape, df_test.shape)
print(df_train.head(3))
print(df_train['label'].value_counts())

(25000, 2) (25000, 2)
                                                text  label
0  Kind of hard to believe that the movie from th...      0
1  I am not a big fan of horror films, and have o...      0
2  Oh a vaguely once famous actress in a film whe...      0
label
0    12500
1    12500
Name: count, dtype: int64


In [ ]:
# 3. 전처리 - TF-IDF 벡터화
df_sample = df_train.sample(5000, random_state=42)

X_text = df_sample['text'].values
y = df_sample['label'].values

tfidf = TfidfVectorizer(stop_words='english',
                        max_features=5000,
                        ngram_range=(1,2))
X = tfidf.fit_transform(X_text)

print('벡터 크기:', X.shape)

벡터 크기: (5000, 5000)


In [ ]:
# 4. 모델 학습 & 평가
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 모델 1: LogisticRegression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

print('Logistic 정확도:', accuracy_score(y_test, log_pred))
print((classification_report(y_test, log_pred, target_names=['부정', '긍정'])))

# 모델 2: NaiveBayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print('NaiveBayes 정확도:', accuracy_score(y_test, nb_pred))

Logistic 정확도: 0.8453333333333334
              precision    recall  f1-score   support

          부정       0.86      0.83      0.84       752
          긍정       0.84      0.86      0.85       748

    accuracy                           0.85      1500
   macro avg       0.85      0.85      0.85      1500
weighted avg       0.85      0.85      0.85      1500

NaiveBayes 정확도: 0.8433333333333334


In [ ]:
# 5. 중요 단어 확인 - 어떤 단어가 긍정/부정을 가르는가
feature_names = tfidf.get_feature_names_out()
coefs = log_model.coef_[0]

top_positive = pd.Series(coefs, index=feature_names).nlargest(10)
print('긍정 키워드:\n', top_positive)

top_negative = pd.Series(coefs, index=feature_names).nsmallest(10)
print('부정 키워드:\n', top_negative)

긍정 키워드:
 great        3.901232
excellent    2.657634
love         2.517171
best         2.435715
wonderful    2.089723
fun          2.059747
perfect      1.918228
beautiful    1.914661
highly       1.885400
enjoyed      1.809934
dtype: float64
부정 키워드:
 worst      -3.799658
bad        -3.425587
boring     -2.844927
awful      -2.776621
waste      -2.770174
poor       -2.333878
terrible   -2.261548
just       -2.226215
reason     -1.987378
acting     -1.905936
dtype: float64


In [ ]:
# 6. 직접 써본 리뷰로 예측
my_reviews = [
    "this movie was absolutely fantastic and the acting was superb",
    "terrible film boring and waste of time awful experience",
    "It was an ambiguous movie"
]
X_my =  tfidf.transform(my_reviews)
print('내 리뷰 예측(1=긍정, 0=부정)', log_model.predict(X_my))

내 리뷰 예측(1=긍정, 0=부정) [1 0 0]


In [ ]:
not_bigrams = [f for f in tfidf.get_feature_names_out() if f.startswith('not')]
print(not_bigrams[:20])

['notable', 'notably', 'notch', 'note', 'noted', 'notes', 'notice', 'noticeable', 'noticed', 'notion', 'notorious']
